# Chapter 2 (hadronic) — Notebook 3: Data vs stacked MC

**Goals**

- Stacked-MC + data plots of $H_T$ and jet multiplicity after the hadronic preselection.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()
samples = io.build_samples()
procs = ['ttbar', 'single_top', 'diboson', 'data']
evs = {p: io.load_process(p, samples, fraction=0.1) for p in procs}
cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)

hists = {}
for p, ev in evs.items():
    sel = selection.semilep_preselection(ev, cuts)
    ht = ak.sum(ev.jet_pt, axis=1)[sel]
    w = weights.weight_for(p, ev)[sel]
    hists[p] = plotting.make_hist(ht, bins=40, range=(0, 800), weight=w, name='HT')

plotting.stacked_plot(hists, xlabel='$H_T$ [GeV]')

## ✏️ Your turn 3.1

▶️ Change the histogram binning/range and re-run.

This builds a stacked-MC + data plot of the invariant mass of the **two leading light**
(non-b-tagged) jets — the hadronic-W candidate. Is there a peak near $m_W \approx 80$ GeV, and does
the real data sit on top of the simulation?

> **Stretch (optional):** lower `MJJ_RANGE`'s upper edge to zoom on the W peak.

In [ ]:
MJJ_BINS  = 40           # ✏️ try 30, 40, 60
MJJ_RANGE = (0, 200)     # ✏️ try (0, 160) to zoom on the W peak

def mjj_light(ev):
    jets = kinematics.jet_vectors(ev)
    light = jets[ev.jet_btag_quantile < cuts.btag_quantile_min]
    keep = ak.num(light) >= 2
    m = (light[keep][:, 0] + light[keep][:, 1]).mass
    return m, keep

hists = {}
for p, ev in evs.items():
    ev_sel = ev[selection.semilep_preselection(ev, cuts)]
    if len(ev_sel) == 0:
        continue
    m, keep = mjj_light(ev_sel)
    w = weights.weight_for(p, ev_sel)[keep]
    hists[p] = plotting.make_hist(m, bins=MJJ_BINS, range=MJJ_RANGE, weight=w, name='mjj')

plotting.stacked_plot(hists, xlabel=r'$m_{jj}$ (two leading light jets) [GeV]')